# Cross-Modal Knowledge Distillation: ConcatMTLFaceRecognitionV2 → MobileNetV3

**Mục tiêu:** Teacher (albedo + normalmap) truyền tri thức đa modal cho student chỉ dùng **một modality duy nhất** lúc inference.

| | Teacher | Student |
|---|---|---|
| Model | `ConcatMTLFaceRecognitionV2` (ConvNeXt V2 × 2) | `FaceRecognitionMobileNetV3` |
| Input | albedo **+** normalmap `[B, 2, 3, 112, 112]` | albedo **hoặc** normalmap `[B, 3, 112, 112]` |
| ID Embedding | 1024-D (512 albedo ⊕ 512 normalmap) | 512-D |
| Mode | **Frozen** | **Trainable** |

**KD Loss:**
```
L_total = α · L_MagFace(student)  +  β · L_KD
L_KD    = mean(1 - cosine_sim(norm(proj(s_emb)), norm(t_emb)))
```

**ProjectionHead:** `Linear(512→1024) → BN1d → ReLU → Linear(1024→1024)` — bridge student (512-D) sang teacher space (1024-D), chỉ dùng khi train.

**Cách dùng:**
1. Chạy cell 1 (Mount Drive)
2. Sửa `CONFIGURATION`, `TEACHER_CKPT_1`, `TEACHER_CKPT_2`, `STUDENT_MODAL_IDX`
3. Chạy từ trên xuống

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

Mounted at /content/drive
/content/FR_Photometric_Stereo
Working dir: /content/FR_Photometric_Stereo


0

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_concatv2_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.ConcatMTLFaceRecognition import ConcatMTLFaceRecognitionV2
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

/content/FR_Photometric_Stereo
Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

# Checkpoint của 2 single-modal teacher đã train xong
TEACHER_CKPT_1 = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'
TEACHER_CKPT_2 = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

# Modality student nhận lúc inference: 0 = albedo (backbone1), 1 = normalmap (backbone2)
STUDENT_MODAL_IDX = 0

EXPERIMENT_NAME = 'KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,
    'output_dir':  '/content/drive/MyDrive/',

    # concat_v2: dataloader trả [B, 2, 3, H, W] (albedo=[:,0], normalmap=[:,1])
    'type':        'concat_v2',

    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',
    'use_sampler': True,
    'device':      device,
    'epochs':      100,
    'num_workers': 2,
    'batch_size':  16,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,

    # L_total = task_weight * L_MagFace + kd_weight * L_KD
    'task_weight': 1.0,
    'kd_weight':   1.0,
}

modal_name = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]
print(f'Student sẽ học từ modal: {modal_name} (index {STUDENT_MODAL_IDX})')
print(f'Teacher ckpt 1 (albedo) : {TEACHER_CKPT_1}')
print(f'Teacher ckpt 2 (normal) : {TEACHER_CKPT_2}')

Student sẽ học từ modal: albedo (index 0)
Teacher ckpt 1 (albedo) : /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth
Teacher ckpt 2 (normal) : /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth


## 3. Data Loading

`create_concatv2_multitask_datafetcher` trả batch `[B, 2, 3, H, W]`:
- `X[:, 0]` = albedo → teacher backbone1 + student input (nếu `STUDENT_MODAL_IDX=0`)
- `X[:, 1]` = normalmap → teacher backbone2 + student input (nếu `STUDENT_MODAL_IDX=1`)

Teacher dùng cả 2, student chỉ dùng `X[:, STUDENT_MODAL_IDX]`.

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(f'Không tìm thấy CSV train tại {dataset_dir}.')
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].max() + 1)
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

# additional_targets={'image2': 'image'} vì concat dataloader dùng key 'image2' cho modal thứ 2
train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
], additional_targets={'image2': 'image'})

test_transform = A.Compose([
    A.Resize(height=CONFIGURATION['image_size'], width=CONFIGURATION['image_size']),
], additional_targets={'image2': 'image'})

train_dl, test_dl = create_concatv2_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

# Kiểm tra shape batch
X_sample, y_sample = next(iter(train_dl))
print(f'Batch shape: X={X_sample.shape}, y={y_sample.shape}')  # [B, 2, 3, 112, 112]

# Gallery-probe loader cho đánh giá cuối
# Cần tạo eval loader cho đúng modality mà student dùng
eval_conf = dict(CONFIGURATION)
eval_conf['type'] = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]
gallery_dl, probe_dl = create_eval_loaders(eval_conf, A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size'])
]))
print(f'Gallery batches: {len(gallery_dl)} | Probe batches: {len(probe_dl)}')

Train CSV: /content/drive/MyDrive/Photometric_DB_Full/train_split.csv
num_classes : 2232
Số mẫu train: 2622
>>> ConcatV2Loader: MODE = PK SAMPLER
Train batches: 163 | Test batches (probe): 18
Batch shape: X=torch.Size([16, 2, 3, 112, 112]), y=torch.Size([16, 6])
Gallery: 68 ảnh | Probe: 288 ảnh
Shared identity space: 68 identities
Gallery batches: 5 | Probe batches: 18


## 4. Teacher Model (ConcatMTLFaceRecognitionV2 — Frozen)

Load 2 single-modal checkpoint đã train, ghép thành `ConcatMTLFaceRecognitionV2`, freeze toàn bộ.

`get_result(X)[0]` trả `id_embedding` **1024-D** (concat từ 2 backbone 512-D mỗi cái).

In [ ]:
for path, name in [(TEACHER_CKPT_1, 'albedo'), (TEACHER_CKPT_2, 'normalmap')]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Không tìm thấy checkpoint {name}: {path}')

def _load_mtl_backbone(ckpt_path, backbone, num_classes, device):
    model = MTLFaceRecognition(backbone=backbone, num_classes=num_classes)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    sd = {k.replace('module.', ''): v for k, v in ckpt['model_state_dict'].items()}
    model.load_state_dict(sd, strict=False)
    print(f'  Loaded: {ckpt_path} (epoch {ckpt.get("epoch", "?")})')
    return model

print('Loading backbone 1 (albedo)...')
mtl_backbone1 = _load_mtl_backbone(
    TEACHER_CKPT_1, CONFIGURATION['teacher_backbone'], CONFIGURATION['num_classes'], device
)

print('Loading backbone 2 (normalmap)...')
mtl_backbone2 = _load_mtl_backbone(
    TEACHER_CKPT_2, CONFIGURATION['teacher_backbone'], CONFIGURATION['num_classes'], device
)

teacher = ConcatMTLFaceRecognitionV2(mtl_backbone1, mtl_backbone2, CONFIGURATION['num_classes'])
teacher.to(device)
teacher.eval()

# Freeze hoàn toàn
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'\nTeacher params: {teacher_params:,} (tất cả frozen)')

# Smoke test — kiểm tra embedding shape
with torch.no_grad():
    _dummy = torch.randn(2, 2, 3, 112, 112).to(device)  # [B, 2, 3, H, W]
    _t_emb = teacher.get_result(_dummy)[0]               # id_embedding: [B, 1024]
    print(f'Teacher ID embedding shape: {_t_emb.shape}')  # mong đợi [2, 1024]

Loading backbone 1 (albedo)...
  Loaded: /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth (epoch 28)
Loading backbone 2 (normalmap)...
  Loaded: /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_NORMALMAP_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth (epoch 96)

Teacher params: 95,561,498 (tất cả frozen)
Teacher ID embedding shape: torch.Size([2, 1024])


## 4.1. Đánh giá Teacher (baseline)

Đo AUC của teacher trên gallery-probe để có baseline so sánh.

In [ ]:
modal_name = ['albedo', 'normalmap'][STUDENT_MODAL_IDX]

# ── Baseline 1: ConvNeXt single-modal (cùng modality với student) ──────────────
class _TeacherSingleModalWrapper(nn.Module):
    """Chỉ dùng 1 backbone của teacher — baseline công bằng với student."""
    def __init__(self, teacher, modal_idx):
        super().__init__()
        self._backbone = teacher.mtl_backbone1 if modal_idx == 0 else teacher.mtl_backbone2

    def get_embedding(self, x):
        return self._backbone.get_embedding(x)[-1]  # [B, 512]


teacher_single = _TeacherSingleModalWrapper(teacher, STUDENT_MODAL_IDX).to(device)
teacher_single.eval()

teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_single, device)
teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_single, device)
print(f'Baseline 1 (single-modal {modal_name}) done.')

# ── Baseline 2: ConcatTeacher full (2 modal) — upper bound ─────────────────────
class _TeacherConcatWrapper(nn.Module):
    """Wrap ConcatMTLFaceRecognitionV2, expose get_embedding() nhận [B, 2, 3, H, W]."""
    def __init__(self, teacher):
        super().__init__()
        self._teacher = teacher

    def get_embedding(self, x):
        # get_result() → (id_embedding 1024-D, gender, pose, emotion, facial_hair, spectacles)
        return self._teacher.get_result(x)[0]  # [B, 1024]


# Gallery/probe loader trả [B, 2, 3, H, W] cho full teacher
concat_eval_conf = dict(CONFIGURATION)
concat_eval_conf['type'] = 'concat_v2'
gallery_concat_dl, probe_concat_dl = create_eval_loaders(
    concat_eval_conf,
    A.Compose([
        A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
    ], additional_targets={'image2': 'image'})
)

teacher_concat = _TeacherConcatWrapper(teacher).to(device)
teacher_concat.eval()

teacher_concat_auc   = compute_id_auc_gallery_probe(gallery_concat_dl, probe_concat_dl, teacher_concat, device)
teacher_concat_rank1 = compute_rank1_gallery_probe(gallery_concat_dl, probe_concat_dl, teacher_concat, device)
print(f'Baseline 2 (full ConcatTeacher, 2 modal) done.')

# ── Bảng tổng hợp 2 baseline ────────────────────────────────────────────────────
compare_rows = [
    ['Input',         f'{modal_name} only',                'albedo + normalmap'],
    ['Embedding dim', '512-D',                             '1024-D'],
    ['Cosine AUC    (gallery→probe)',
     f"{teacher_gp_auc['id_cosine']:.4f}",
     f"{teacher_concat_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)',
     f"{teacher_gp_auc['id_euclidean']:.4f}",
     f"{teacher_concat_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)',
     f"{teacher_gp_rank1:.4f}",
     f"{teacher_concat_rank1:.4f}"],
]
print(f"\n--- Teacher Baselines ({CONFIGURATION['teacher_backbone']}) ---")
print(tabulate(compare_rows,
               headers=['Metric', 'Single-modal (baseline)', 'Full teacher (upper bound)'],
               tablefmt='fancy_grid'))

Baseline 1 (single-modal albedo) done.
Gallery: 68 ảnh | Probe: 288 ảnh
Shared identity space: 68 identities
Baseline 2 (full ConcatTeacher, 2 modal) done.

--- Teacher Baselines (convnextv2_tiny) ---
╒═══════════════════════════════╤═══════════════════════════╤══════════════════════════════╕
│ Metric                        │ Single-modal (baseline)   │ Full teacher (upper bound)   │
╞═══════════════════════════════╪═══════════════════════════╪══════════════════════════════╡
│ Input                         │ albedo only               │ albedo + normalmap           │
├───────────────────────────────┼───────────────────────────┼──────────────────────────────┤
│ Embedding dim                 │ 512-D                     │ 1024-D                       │
├───────────────────────────────┼───────────────────────────┼──────────────────────────────┤
│ Cosine AUC    (gallery→probe) │ 0.9761                    │ 0.9825                       │
├───────────────────────────────┼──────────────────────

## 5. Student Model (MobileNetV3 — Trainable)

Nhận **1 modality** `[B, 3, 112, 112]`, output embedding **512-D**.

In [ ]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

# Smoke test
_dummy_single = torch.randn(2, 3, 112, 112).to(device)
with torch.no_grad():
    _s_emb = student.get_embedding(_dummy_single)
    print(f'Student embedding shape: {_s_emb.shape}')  # mong đợi [2, 512]

model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Student total params    : 4,608,304
Student trainable params: 4,608,304
Student embedding shape: torch.Size([2, 512])


## 6. Projection Head (512-D → 1024-D)

Bridge student space (512-D) sang teacher space (1024-D) để tính KD loss.

```
student_emb (512) → Linear(512→1024) → BN1d → ReLU → Linear(1024→1024) → proj_emb (1024)
                                                                              ↕ cosine KD loss
                                                             teacher_id_emb (1024)
```

- `student_emb` vẫn đi thẳng vào `MagLinear` (task loss không thay đổi)
- Projector **chỉ tồn tại khi train**, bỏ hoàn toàn khi export ONNX

In [ ]:
class ProjectionHead(nn.Module):
    """
    MLP bridge: student embedding space (512) → teacher embedding space (1024).
    Chỉ dùng khi tính KD loss, không tham gia inference / ONNX export.
    """
    def __init__(self, in_dim: int = 512, hidden_dim: int = 1024, out_dim: int = 1024):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(x)


projector = ProjectionHead(in_dim=512, hidden_dim=1024, out_dim=1024).to(device)
proj_params = sum(p.numel() for p in projector.parameters())
print(f'ProjectionHead params: {proj_params:,}  (train-only, dropped at export)')

# Smoke test
with torch.no_grad():
    _proj_emb = projector(_s_emb)
    print(f'Projected embedding shape: {_proj_emb.shape}')  # mong đợi [2, 1024]

ProjectionHead params: 1,576,960  (train-only, dropped at export)
Projected embedding shape: torch.Size([2, 1024])


## 7. Knowledge Distillation Loss

```
L_total = α · L_MagFace(student_emb, id_labels)
        + β · L_KD_cosine(proj(student_emb), teacher_id_emb)
```

**Tại sao cosine loss:** face verification so sánh *hướng* embedding, không phải magnitude.  
Cosine loss kéo student về đúng hướng mà teacher biết từ thông tin đa modal.

In [ ]:
class CrossModalKDLoss(nn.Module):
    """
    L_total = task_weight * L_MagFace  +  kd_weight * L_KD_cosine

    student_emb  : [B, 512]  — embedding trước MagLinear (task loss)
    proj_emb     : [B, 1024] — student_emb sau ProjectionHead (KD loss)
    teacher_emb  : [B, 1024] — fused ID embedding của ConcatTeacher (no_grad)
    """

    def __init__(self, metadata_path: str, task_weight: float = 1.0, kd_weight: float = 1.0):
        super().__init__()
        self.magface = WeightClassMagLoss(metadata_path)
        self.task_w  = task_weight
        self.kd_w    = kd_weight

    def forward(
        self,
        student_logits,
        student_norm,
        student_emb,
        proj_emb,
        teacher_emb,
        id_labels,
    ):
        l_task = self.magface(student_logits, id_labels, student_norm)

        # KD loss trên unit hypersphere 1024-D
        p_n = F.normalize(proj_emb,    p=2, dim=1)
        t_n = F.normalize(teacher_emb, p=2, dim=1)
        l_kd = (1.0 - F.cosine_similarity(p_n, t_n, dim=1)).mean()

        total = self.task_w * l_task + self.kd_w * l_kd
        return total, l_task, l_kd


criterion = CrossModalKDLoss(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
)
print('CrossModalKDLoss khởi tạo thành công.')

CrossModalKDLoss khởi tạo thành công.


## 8. Training

In [ ]:
def train_epoch(train_dl, teacher, student, projector, criterion, optimizer, device, student_modal_idx):
    student.train()
    projector.train()
    # teacher.eval() + frozen — không cần đặt lại mỗi epoch

    total_loss = total_task = total_kd = 0.0

    for X, y in train_dl:
        # X shape: [B, 2, 3, H, W]  (axis 1: 0=albedo, 1=normalmap)
        X, y = X.to(device), y.to(device)
        id_labels = y[:, 0]

        # Teacher nhận cả 2 modality — lấy fused ID embedding 1024-D
        with torch.no_grad():
            teacher_emb = teacher.get_result(X)[0]  # [B, 1024]

        # Student chỉ nhận 1 modality
        X_student = X[:, student_modal_idx]  # [B, 3, H, W]

        feat        = student.backbone(X_student)     # [B, 512, H', W']
        student_emb = student.embedding(feat)         # [B, 512]
        proj_emb    = projector(student_emb)          # [B, 1024] — bridge to teacher space
        logits, norm = student.maglinear(student_emb)

        loss, l_task, l_kd = criterion(
            logits, norm, student_emb, proj_emb, teacher_emb, id_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_task += l_task.item()
        total_kd   += l_kd.item()

    n = len(train_dl)
    return total_loss / n, total_task / n, total_kd / n


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
# Optimizer update cả student lẫn projector
optimizer = Adam(
    list(student.parameters()) + list(projector.parameters()),
    lr=CONFIGURATION['base_lr'],
)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Teacher: ConcatMTLFaceRecognitionV2 ({CONFIGURATION['teacher_backbone']} x2) | "
    f"Student: {CONFIGURATION['backbone']} | "
    f"Student modal: {['albedo','normalmap'][STUDENT_MODAL_IDX]} (idx={STUDENT_MODAL_IDX}) | "
    f"Teacher emb: 1024-D | Student emb: 512-D | "
    f"kd_weight={CONFIGURATION['kd_weight']} task_weight={CONFIGURATION['task_weight']} | "
    f"projector=True (512→1024)"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=10,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')

KHOI TAO THI NGHIEM: KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo
Luu tru tai: /content/drive/MyDrive/experiments/KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo
Thoi gian: 2026-06-02 16:16:15
--------------------------------------------------
Teacher: ConcatMTLFaceRecognitionV2 (convnextv2_tiny x2) | Student: mobilenetv3_large_100 | Student modal: albedo (idx=0) | Teacher emb: 1024-D | Student emb: 512-D | kd_weight=1.0 task_weight=1.0 | projector=True (512→1024)
Experiment dir: /content/drive/MyDrive/experiments/KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo
Checkpoint dir: /content/drive/MyDrive/experiments/KD_CrossModal_ConcatTeacher_to_MobileNetV3_Albedo/checkpoints


In [ ]:
START_EPOCH = 0

manager.log_text('BAT DAU CROSS-MODAL KNOWLEDGE DISTILLATION')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    train_loss, train_task, train_kd = train_epoch(
        train_dl, teacher, student, projector, criterion,
        optimizer, device, STUDENT_MODAL_IDX,
    )

    # AUC dùng student.get_embedding() trực tiếp (không qua projector)
    # eval_dl chỉ chứa 1 modality (đúng với STUDENT_MODAL_IDX)
    train_auc = compute_id_auc(train_dl, student, device,
                               modal_idx=STUDENT_MODAL_IDX)  # truyền idx nếu loader trả [B,2,...]
    test_auc  = compute_id_auc(test_dl,  student, device,
                               modal_idx=STUDENT_MODAL_IDX)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd':          train_kd,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    writer.add_scalar('Loss/total', train_loss, epoch + 1)
    writer.add_scalar('Loss/task',  train_task, epoch + 1)
    writer.add_scalar('Loss/kd',    train_kd,   epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver(
        student, optimizer, epoch + 1, test_metrics, scheduler,
        extra_state={'projector_state_dict': projector.state_dict()},
    )
    early_stopping(test_metrics, student, epoch + 1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('CROSS-MODAL KNOWLEDGE DISTILLATION HOAN TAT.')

BAT DAU CROSS-MODAL KNOWLEDGE DISTILLATION

--- Epoch 1/100 ---

Ep 1:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 32.3766 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_task        │ 31.5224 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_kd          │  0.8541 │ -      │
├──────────────────┼─────────┼────────┤
│ auc_id_cosine    │  0.9207 │ 0.8938 │
├──────────────────┼─────────┼────────┤
│ auc_id_euclidean │  0.9207 │ 0.8938 │
╘══════════════════╧═════════╧════════╛
Ep 1: loss: 32.3766, loss_task: 31.5224, loss_kd: 0.8541, auc_id_cosine: 0.8938, auc_id_euclidean: 0.8938
--> SAVE BEST MODEL (auc_id_cosine: 0.8938)

--- Epoch 2/100 ---

Ep 2:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 32.599  │ -      │
├──────────────────┼─────────┼────────┤
│ loss_task        │ 31.803  │ 

## 9. Resume Training từ Checkpoint

> Chạy khi Colab disconnect.  
> **Cách dùng:** Chạy cell Setup → Imports → Data → Teacher → Student → Projector → Loss → Setup Train,  
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

checkpoint = torch.load(CKPT_PATH, map_location=device)
student.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
if 'projector_state_dict' in checkpoint:
    projector.load_state_dict(checkpoint['projector_state_dict'])
    print('Projector state loaded.')
else:
    print('Không tìm thấy projector_state_dict — projector khởi tạo ngẫu nhiên.')

START_EPOCH = checkpoint['epoch']
print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

## 10. Đánh giá Final

So sánh:
- **Teacher (single modal baseline):** ConvNeXt backbone dùng đúng modality đó, không có cross-modal
- **Student (cross-modal KD):** MobileNetV3 học từ fused teacher 1024-D

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

# 3 cột: ConvNeXt single-modal | Full ConcatTeacher | Student sau KD
compare_rows = [
    ['Model',
     f"{CONFIGURATION['teacher_backbone']} ({modal_name} only)",
     f"ConcatTeacher ({CONFIGURATION['teacher_backbone']} ×2)",
     f"{CONFIGURATION['backbone']} (cross-modal KD)"],
    ['Input',
     f'{modal_name} only',
     'albedo + normalmap',
     f'{modal_name} only'],
    ['Params',
     f'~{teacher_params // 2 // 1_000_000}M (est.)',
     f'~{teacher_params // 1_000_000}M',
     f'~{total_p // 1_000_000}M'],
    ['Cosine AUC (gallery→probe)',
     f"{teacher_gp_auc['id_cosine']:.4f}",
     f"{teacher_concat_auc['id_cosine']:.4f}",
     f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)',
     f"{teacher_gp_auc['id_euclidean']:.4f}",
     f"{teacher_concat_auc['id_euclidean']:.4f}",
     f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc (gallery→probe)',
     f"{teacher_gp_rank1:.4f}",
     f"{teacher_concat_rank1:.4f}",
     f"{student_gp_rank1:.4f}"],
]
print('\n--- So sánh kết quả ---')
print(tabulate(compare_rows,
               headers=['Metric', 'Single-modal baseline', 'Upper bound (2 modal)', 'Student (cross-modal KD)'],
               tablefmt='fancy_grid'))

Best model từ epoch 54

--- So sánh kết quả ---
╒═══════════════════════════════╤═══════════════════════════════╤════════════════════════════════════╤════════════════════════════════════════╕
│ Metric                        │ Single-modal baseline         │ Upper bound (2 modal)              │ Student (cross-modal KD)               │
╞═══════════════════════════════╪═══════════════════════════════╪════════════════════════════════════╪════════════════════════════════════════╡
│ Model                         │ convnextv2_tiny (albedo only) │ ConcatTeacher (convnextv2_tiny ×2) │ mobilenetv3_large_100 (cross-modal KD) │
├───────────────────────────────┼───────────────────────────────┼────────────────────────────────────┼────────────────────────────────────────┤
│ Input                         │ albedo only                   │ albedo + normalmap                 │ albedo only                            │
├───────────────────────────────┼───────────────────────────────┼───────────────────────

## 11. Export ONNX

Export phần inference (backbone + embedding + L2 normalize, bỏ MagLinear và projector)  
để chuẩn bị quantize INT8 deploy edge device.

In [ ]:
class InferenceWrapper(nn.Module):
    """Backbone + embedding + L2 normalize — không có MagLinear, không có Projector."""
    def __init__(self, model):
        super().__init__()
        self.backbone  = model.backbone
        self.embedding = model.embedding

    def forward(self, x):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb, p=2, dim=1)


inference_model = InferenceWrapper(student).eval().cpu()
dummy_input = torch.randn(1, 3, 112, 112)

onnx_path = os.path.join(manager.ckpt_dir, 'kd_crossmodal_mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')
print(f'Input : [B, 3, 112, 112] — chỉ cần {modal_name}')
print(f'Output: [B, 512] — L2-normalized embedding')